In [1]:
#Modules
%config InlineBackend.figure_format='retina'
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from astropy.io import fits
import pandas as pd
from io import StringIO
from matplotlib.path import Path
import os
from astropy.wcs import WCS
from astropy import units as u
from astropy.coordinates import SkyCoord
from shapely.geometry import Polygon
from astropy.utils.data import get_pkg_data_filename
from astropy.wcs import Wcsprm
from astropy.io import fits
from astropy.wcs import utils
from skimage.feature import blob_dog, blob_log, blob_doh
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import colors
from skimage.color import rgb2gray, rgb2hsv, hsv2rgb
from skimage.io import imread, imshow
from sklearn.cluster import KMeans
from astropy.visualization import make_lupton_rgb
from astropy.visualization import SqrtStretch,LogStretch
from astropy.visualization import ZScaleInterval
from reproject import reproject_interp
from astropy.visualization import make_lupton_rgb
import sep
from matplotlib.patches import Ellipse
from scipy import stats
from scipy.ndimage import gaussian_filter
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from scipy.interpolate import griddata

from scipy.stats import kde
from scipy.ndimage import gaussian_filter
from astropy.io import ascii
from astropy.visualization import simple_norm
from photutils.psf import extract_stars


from astropy.table import Table
from astropy.nddata import NDData

from shapely.geometry import Polygon, Point
from photutils.detection import find_peaks

from skimage.measure import EllipseModel

from astropy.stats import SigmaClip

from photutils.background import Background2D, MedianBackground
from scipy import interpolate
ell = EllipseModel()

image_lims=ZScaleInterval()

plt.rcParams['font.size'] = 15
from scipy.interpolate import LinearNDInterpolator
from scipy.interpolate import CloughTocher2DInterpolator

from astropy.modeling.models import Gaussian2D
from astropy.modeling.models import custom_model
from astropy.modeling import models, fitting
from scipy.optimize import curve_fit

from photutils.segmentation import SegmentationImage
from astropy.stats import sigma_clipped_stats, SigmaClip
from photutils.segmentation import detect_threshold, detect_sources

from astropy.convolution import convolve, Gaussian2DKernel, Gaussian1DKernel
from astropy.convolution import interpolate_replace_nans

from photutils.segmentation import deblend_sources, detect_threshold, detect_sources

from photutils.segmentation import SourceFinder
from photutils.segmentation import SourceCatalog

from astropy.modeling.fitting import LevMarLSQFitter
from astropy.stats import gaussian_sigma_to_fwhm
from photutils.background import MADStdBackgroundRMS, MMMBackground
from photutils.detection import IRAFStarFinder
from photutils.psf import (DAOGroup, IntegratedGaussianPRF,
                           IterativelySubtractedPSFPhotometry,BasicPSFPhotometry,prepare_psf_model)

from photutils.datasets import make_noise_image
from photutils.psf import EPSFBuilder
from photutils.utils import circular_footprint
from tempfile import TemporaryFile
from astropy.io import fits as pyfits
import pickle

In [2]:
#Open vertice coordinate files

#Use IRAC coordinates
with open(r'C:\Users\jacob\OneDrive\Desktop\Cluster_Project\jw_verts3_conservative.pkl', 'rb') as fp1:
    vertices1 = pickle.load(fp1)
    
with open(r'C:\Users\jacob\OneDrive\Desktop\Cluster_Project\jw_verts3_expansive.pkl', 'rb') as fp3:
    vertices3 = pickle.load(fp3)
    
    
#Use VMC coordinates
with open(r"C:\Users\jacob\OneDrive\Desktop\Cluster_Project\verts_conservative_higher_6MARCH.pkl", 'rb') as fp25:
    vertices25 = pickle.load(fp25)
    
with open(r"C:\Users\jacob\OneDrive\Desktop\Cluster_Project\verts_expansive_higher_6MARCH.pkl", 'rb') as fp45:
    vertices45 = pickle.load(fp45)

In [3]:
#Open interpolated 24 micron file

image24 = np.load(r"C:\Users\jacob\OneDrive\Desktop\Cluster_Project\24_image_reprojected.npy")
backsub24 = np.load(r"C:\Users\jacob\OneDrive\Desktop\Cluster_Project\24_bkg_reprojected.npy")

In [4]:
#Open a VMC file
hdux = fits.open(r"C:\Users\jacob\OneDrive\Desktop\Cluster_Project\e20230725_00130000217_dp_st_tl.fit") 

In [5]:
#Constants and Conversions

zpt24 = 7.25
pix2arcsec = 0.34
arcsec2degree = 1./3600.
sr2squaredegree = 4.*np.pi/41253.
MJy2Jy = 1000000


In [48]:
#Create a blank dictionary to store magnitudes
magnitudes3 = {}
magnitudes3['key'] = []
magnitudes3['24'] = []

In [5]:
#Computes medians as needed

sigma_clip = SigmaClip(sigma = 3.0, maxiters = 10)

threshold1 = detect_threshold(image24, nsigma = 2.0, sigma_clip = sigma_clip)

segment_img1 = detect_sources(image24, threshold1, npixels = 10)

footprint = circular_footprint(radius = 10)

mask1 = segment_img1.make_source_mask(footprint = footprint)

mean, median11, std = sigma_clipped_stats(image24, sigma=3.0, mask = mask1)

np.save('24BacksubMedianValue', median11)

print(median11)

0.8705007430689855


In [19]:
median24 = np.load('24BacksubMedianValue.npy')

In [ ]:
#Conservative 24 Micron Plots

q = 0
ii = 0
backsub1 = backsub24
while q < 1:

    u = 0
    for key in vertices1:

        array = vertices1.get(key)
        ra = array[0]
        dec = array[1]

        xpixels, ypixels = WCS(hdux[1].header).all_world2pix(ra,dec, 1)
        xmin = np.min(xpixels)
        xmax = np.max(xpixels)
        ymin = np.min(ypixels)
        ymax = np.max(ypixels)
        cutout = backsub1[int(np.round(ymin)):int(np.round(ymax)), int(np.round(xmin)):int(np.round(xmax))]
        vmin,vmax = image_lims.get_limits(cutout)

        levels2 = np.arange(5,200,10)
        plt.figure(figsize = (16,8))
        ax = plt.subplot(111)
        im= ax.imshow(cutout,cmap = 'gray_r',vmin = vmin,vmax = vmax)
        plt.plot(xpixels-int(np.round(xmin)), ypixels-int(np.round(ymin)))
        ax.set_title('Contour ' + str(ii) + ' 3.8 Micron')

        ii = ii+1
        u = u+1
        if u == 50:
            break
    break

In [ ]:
#Loops for the interpolated images
#This is for the conservative and expansive vertices sets
#Vertices numbers should be 1 or 3


ran = [0]

for ii in ran:
    if ii == 0:
        jj = 1
        
        #24 Micron and Keys
        
        backsub24 = backsub24
        #backsub24 = image24 - median24
        

        for key in vertices3:
            
            array = vertices3.get(key)

            ra = array[0]
            dec = array[1]


            magnitudes3['key'].append(key)


            print('These are the 24 micron magnitudes of Contour ' + str(jj))

            xpixels, ypixels = WCS(hdux[1].header).all_world2pix(ra,dec, 1)
            xmin = np.min(xpixels)-1
            xmax = np.max(xpixels)+1
            ymin = np.min(ypixels)-1
            ymax = np.max(ypixels)+1


            #24 Micron Magnitude
            cutout = backsub24[int(np.round(ymin)-1):int(np.round(ymax)-1), int(np.round(xmin)-1):int(np.round(xmax)-1)]
            flat = cutout.flatten()
            xv,yv = np.indices(cutout.shape)
            flaty = xv.flatten()
            flatx = yv.flatten()
            points = np.vstack((flatx,flaty)).T
            polygon = Path(np.stack((xpixels-xmin, ypixels-ymin), axis = -1 ))
            grid = polygon.contains_points(points)
            index_true, = np.where(grid == True)
            index2 = np.reshape(grid, cutout.shape)
            copy = cutout.copy()
            copy[index2] = -9000
            sumAll = sum(cutout.flatten())

            sumInside = sum(cutout[index2])
            
            initialFluxDensity1 = sumInside
            convertedFluxDensity1 = initialFluxDensity1*pix2arcsec**2*arcsec2degree**2*sr2squaredegree*MJy2Jy   
            convertedFluxDensity1 = float(convertedFluxDensity1)
            mag1 = (2.5*np.log10(zpt24/convertedFluxDensity1))  
            print(mag1)
            
            
            magnitudes3['24'].append(mag1)
            
            jj = jj+1


In [ ]:
#Loops for the interpolated images
#Vertices numbers should be  25 or 45
#No ECF will be used
#This is for the higher vertices sets


ran = [0]

for ii in ran:
    if ii == 0:
        jj = 1
        
        #24 Micron and Keys
        
        backsub24 = backsub24
        #backsub124 = image24 - median24
        

        for key in vertices45:
            
            array = vertices45.get(key)

            pixx1 = array[:,0]
            pixy1 = array[:,1]

            ra, dec = WCS(hdux[1].header).all_pix2world(pixx1,pixy1, 1)


            magnitudes3['key'].append(key)


            print('These are the 24 magnitudes of Contour ' + str(jj))

            xpixels, ypixels = WCS(hdux[1].header).all_world2pix(ra,dec, 1)
            xmin = np.min(xpixels)-1
            xmax = np.max(xpixels)+1
            ymin = np.min(ypixels)-1
            ymax = np.max(ypixels)+1


            #3.6 Micron Magnitude
            cutout = backsub24[int(np.round(ymin)-1):int(np.round(ymax)-1), int(np.round(xmin)-1):int(np.round(xmax)-1)]
            flat = cutout.flatten()
            xv,yv = np.indices(cutout.shape)
            flaty = xv.flatten()
            flatx = yv.flatten()
            points = np.vstack((flatx,flaty)).T
            polygon = Path(np.stack((xpixels-xmin, ypixels-ymin), axis = -1 ))
            grid = polygon.contains_points(points)
            index_true, = np.where(grid == True)
            index2 = np.reshape(grid, cutout.shape)
            copy = cutout.copy()
            copy[index2] = -9000
            sumAll = sum(cutout.flatten())

            sumInside = sum(cutout[index2])

           
            initialFluxDensity1 = sumInside
            convertedFluxDensity1 = initialFluxDensity1*pix2arcsec**2*arcsec2degree**2*sr2squaredegree*MJy2Jy   
            convertedFluxDensity1 = float(convertedFluxDensity1)
            mag1 = (2.5*np.log10(zpt24/convertedFluxDensity1))  
            print(mag1)
            
            
            magnitudes3['24'].append(mag1)

            jj = jj+1


In [50]:
#Create table using the magnitude dictionary and save as a FITS file


magnitudes3_table = Table(magnitudes3)
print(magnitudes3_table)
magnitudes3_table.write('24Micron_magnitudes(interpolated_Amy_BKG_subtraction)_expansive_higher.fits')

     key              24       
------------- -----------------
  111058_10_0               nan
  111058_15_0               nan
  111058_20_0               nan
  111058_25_0               nan
  111058_30_0               nan
  322939_10_1               nan
  322939_15_1               nan
  381676_10_0 6.944222580046051
  381676_10_3 6.347652269881534
  381676_15_4 6.767356149701497
          ...               ...
1388810_20_12               nan
1388810_25_12               nan
1388810_30_14               nan
1388810_35_13               nan
1391848_10_11               nan
1391848_10_26               nan
1391848_15_32               nan
1391848_20_31               nan
1391848_25_32               nan
1391848_30_31               nan
1391848_35_28               nan
Length = 27361 rows
